# 🌿 Prédiction du Green-Score (Eco-Score) avec Embeddings & Vecteurs

Ce notebook entraîne un modèle ML pour prédire le **Green-Score** de produits alimentaires à partir des données OpenFoodFacts.

## Méthodologie Green-Score
Le Green-Score = **score ACV normalisé /100 + bonus/malus** :
- 🔬 **Score de base** : ACV (Analyse du Cycle de Vie) via Agribalyse → normalisé sur 100
- ✅ **Bonus** : Labels bio (+10/+20 pts), Saisonnalité (+5 pts), Origine (+3 pts max)
- ❌ **Malus** : Espèces menacées (-10 pts), Emballage non recyclable (-10 pts max), Origine (-7 pts max)

**Lettres** : A+ (≥80) > A (≥70) > B (≥55) > C (≥40) > D (≥25) > E (≥10) > F (<10)

## Pipeline
1. Chargement & nettoyage des données OpenFoodFacts
2. Feature engineering (catégorie, labels, pays, emballage, ingrédients)
3. Embeddings textuels : TF-IDF + Sentence Transformers
4. Modèles : Régression (score numérique) + Classification (grade A→F)
5. Évaluation & analyse des features importantes
6. Inférence sur de nouveaux produits

## 0. Installation des dépendances

In [ ]:
!pip install pandas numpy scikit-learn lightgbm sentence-transformers matplotlib seaborn tqdm pyarrow fastparquet -q

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, r2_score, mean_squared_error,
    classification_report, confusion_matrix, accuracy_score
)
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from scipy.sparse import hstack, csr_matrix

import lightgbm as lgb
from tqdm import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')

# ── Configuration ──────────────────────────────────────────────────────────
RANDOM_STATE = 42
CHUNK_SIZE   = 20_000   # Taille de chaque bloc lu en mémoire
MAX_ROWS     = None     # None = tout lire | ex: 500_000 pour limiter

# Labels bio et environnementaux → bonus Green-Score
POSITIVE_LABELS = [
    'en:organic', 'en:eu-organic', 'fr:ab-agriculture-biologique',
    'en:rainforest-alliance', 'en:fairtrade-international',
    'en:demeter', 'en:biodynamic', 'en:sustainable-seafood',
    'en:roundtable-on-sustainable-palm-oil-rspo',
    'en:high-environmental-value', 'en:hve'
]

# Poissons menacés → malus
ENDANGERED_FISH_KEYWORDS = [
    'thon', 'tuna', 'anguille', 'eel', 'espadon', 'swordfish',
    'requin', 'shark', 'cabillaud', 'cod', 'flétan', 'halibut',
    'bar', 'sea bass', 'dorade', 'sea bream'
]

# Emballages non recyclables → malus
BAD_PACKAGING = [
    'en:non-recyclable-and-non-biodegradable-material',
    'en:plastic', 'en:polystyrene'
]
GOOD_PACKAGING = [
    'en:recycled-material', 'en:recyclable-material',
    'en:reusable-packaging', 'en:compostable'
]

print("✅ Imports OK")

## 2. Chargement des données OpenFoodFacts (streaming par chunks)

> **Aucun téléchargement local** : `pandas` lit le `.csv.gz` directement depuis l'URL OpenFoodFacts via HTTP.
> Seules les colonnes utiles sont sélectionnées avec `usecols`, et chaque chunk est nettoyé **avant** d'être conservé en mémoire.
> La fonction `process_data` suit la même convention que le pipeline Nutriscore.

In [ ]:
# ── Chemin source (URL directe, pas de téléchargement local) ──────────────
INPUT_PATH = "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"

# ── Colonnes cibles ────────────────────────────────────────────────────────
TARGET_SCORE = 'ecoscore_score'   # cible numérique (régression)
TARGET_GRADE = 'ecoscore_grade'   # cible catégorielle (classification)

IDENTITY_COLS = ['code', 'product_name']

# Colonnes textuelles & catégorielles pour les embeddings
TEXT_COLS = [
    'pnns_groups_1', 'pnns_groups_2',
    'categories', 'categories_tags',
    'main_category',
    'ingredients_text',
    'labels', 'labels_tags',
    'countries', 'countries_tags',
    'origins', 'origins_tags',
    'manufacturing_places',
    'packaging', 'packaging_tags',
]

# Colonne ACV Agribalyse (proxy du single score)
NUMERIC_COLS = ['agribalyse_score']

# Liste complète passée à usecols
ALL_COLS = IDENTITY_COLS + [TARGET_SCORE, TARGET_GRADE] + TEXT_COLS + NUMERIC_COLS

# Grades valides selon la méthodologie Green-Score
VALID_GRADES = {'a-plus', 'a', 'b', 'c', 'd', 'e', 'f'}


# ── Fonction de chargement par chunks (style nutriscore) ───────────────────
def process_data(file_path: str, cols: list, chunk_size: int = 20_000,
                 max_rows: int = None) -> pd.DataFrame:
    """
    Lit le CSV OpenFoodFacts directement depuis `file_path` (URL ou chemin local)
    par blocs de `chunk_size` lignes, en ne gardant que `cols`.

    Nettoyages appliqués par chunk :
      1. Suppression des lignes sans ecoscore_score / ecoscore_grade
      2. Filtrage des grades invalides (hors A+/A/B/C/D/E/F)
      3. Conversion numérique du score + filtre de réalisme [0, 100]
      4. Conversion numérique d'agribalyse_score
      5. Normalisation du grade en minuscules
      6. Remplissage des colonnes textuelles manquantes par ''
      7. Déduplication sur 'code'
    """
    reader = pd.read_csv(
        file_path,
        compression='gzip',
        sep='\t',
        on_bad_lines='skip',
        chunksize=chunk_size,
        low_memory=False,
        usecols=lambda c: c in cols,
        dtype={'code': str},
    )

    clean_chunks = []
    total_kept = 0
    print("📖 Lecture des blocs depuis l'URL...")

    for i, chunk in enumerate(reader):

        # 1. Supprimer les lignes sans cible
        chunk = chunk.dropna(subset=[TARGET_SCORE, TARGET_GRADE])
        if chunk.empty:
            continue

        # 2. Filtrer les grades invalides
        chunk[TARGET_GRADE] = chunk[TARGET_GRADE].str.lower().str.strip()
        chunk = chunk[chunk[TARGET_GRADE].isin(VALID_GRADES)]
        if chunk.empty:
            continue

        # 3. Conversion + filtre de réalisme du score [0, 100]
        chunk[TARGET_SCORE] = pd.to_numeric(chunk[TARGET_SCORE], errors='coerce')
        chunk = chunk.dropna(subset=[TARGET_SCORE])
        chunk = chunk[
            (chunk[TARGET_SCORE] >= 0) & (chunk[TARGET_SCORE] <= 100)
        ]
        if chunk.empty:
            continue

        # 4. Conversion numérique d'agribalyse_score
        if 'agribalyse_score' in chunk.columns:
            chunk['agribalyse_score'] = pd.to_numeric(
                chunk['agribalyse_score'], errors='coerce'
            ).fillna(-1.0)
        else:
            chunk['agribalyse_score'] = -1.0

        # 5. Remplissage des colonnes textuelles manquantes
        for col in TEXT_COLS:
            if col in chunk.columns:
                chunk[col] = chunk[col].fillna('')
            else:
                chunk[col] = ''

        # 6. Identité
        chunk['product_name'] = chunk['product_name'].fillna('Unknown Product')

        # 7. Déduplication
        chunk = chunk.drop_duplicates(subset=['code'])

        clean_chunks.append(chunk)
        total_kept += len(chunk)

        if i % 10 == 0:
            print(f"  Bloc {i:>4} | {total_kept:>8,} produits valides conservés")

        # Limite optionnelle
        if max_rows is not None and total_kept >= max_rows:
            print(f"  ⛔ Limite de {max_rows:,} lignes atteinte, arrêt.")
            break

    print("\n🔗 Fusion de tous les blocs...")
    df_final = pd.concat(clean_chunks, ignore_index=True)

    # Déduplication globale (doublons à cheval sur deux chunks)
    df_final = df_final.drop_duplicates(subset=['code'])

    return df_final


# ── Lancement ──────────────────────────────────────────────────────────────
df_raw = process_data(INPUT_PATH, ALL_COLS, chunk_size=CHUNK_SIZE, max_rows=MAX_ROWS)

print(f"\n✅ Dataset final : {len(df_raw):,} produits × {df_raw.shape[1]} colonnes")
print(f"   Grades présents : {sorted(df_raw[TARGET_GRADE].unique())}")
df_raw.head(3)

## 3. Exploration & Analyse de la cible

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution du score numérique
axes[0].hist(df_raw['ecoscore_score'].astype(float), bins=50,
             color='#2ecc71', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribution du Green-Score (numérique)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Score /100')
axes[0].set_ylabel('Nombre de produits')
axes[0].axvline(df_raw['ecoscore_score'].astype(float).mean(), color='red',
                linestyle='--', label=f"Moyenne = {df_raw['ecoscore_score'].astype(float).mean():.1f}")
axes[0].legend()

# Distribution des grades
grade_order = ['a-plus', 'a', 'b', 'c', 'd', 'e', 'f']
grade_colors = ['#00b050', '#57b544', '#a8c832', '#ffcd00', '#ff8c00', '#e02020', '#a00000']
grade_counts = df_raw['ecoscore_grade'].str.lower().value_counts()
grade_counts = grade_counts.reindex([g for g in grade_order if g in grade_counts.index])
bars = axes[1].bar(grade_counts.index, grade_counts.values,
                   color=grade_colors[:len(grade_counts)])
axes[1].set_title('Distribution des Grades Green-Score', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Grade')
axes[1].set_ylabel('Nombre de produits')
for bar, val in zip(bars, grade_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('ecoscore_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Statistiques du score :")
print(df_raw['ecoscore_score'].astype(float).describe().round(2))

## 4. Feature Engineering

On construit les features en suivant la **méthodologie Green-Score** :
- Features textuelles pour les embeddings (nom, catégorie, ingrédients, labels)
- Features structurées (bonus/malus explicites)
- Score Agribalyse si disponible

In [ ]:
def safe_str(x):
    """Convertit en string propre ou retourne empty string."""
    if pd.isna(x):
        return ''
    return str(x).lower().strip()


def extract_features(df: pd.DataFrame) -> pd.DataFrame:
    """Construit toutes les features selon la méthodologie Green-Score."""
    out = pd.DataFrame(index=df.index)

    # ── Cibles ────────────────────────────────────────────────────────────
    out['ecoscore_score'] = pd.to_numeric(df['ecoscore_score'], errors='coerce')
    out['ecoscore_grade'] = df['ecoscore_grade'].str.lower().str.strip()

    # ── Texte combiné pour embeddings ────────────────────────────────────
    out['text_product']     = df['product_name'].apply(safe_str)
    out['text_categories']  = df['categories'].apply(safe_str)
    out['text_ingredients'] = df['ingredients_text'].apply(safe_str)
    out['text_labels']      = df['labels'].apply(safe_str)
    out['text_packaging']   = df['packaging'].apply(safe_str)
    out['text_countries']   = df['countries'].apply(safe_str)
    out['text_origins']     = df['origins'].apply(safe_str)

    # Texte enrichi (concaténation pour le modèle d'embedding principal)
    out['text_rich'] = (
        out['text_product'] + ' '
        + out['text_categories'] + ' '
        + out['text_labels'] + ' '
        + out['text_ingredients'].apply(lambda x: x[:300])  # tronquer les listes longues
    ).str.strip()

    # ── 1. Labels bio / environnementaux → bonus +10 à +20 pts ────────────
    labels_tags = df['labels_tags'].apply(safe_str)
    out['has_organic_label'] = labels_tags.str.contains(
        'organic|biologique|bio|demeter|biodynamic', regex=True
    ).astype(int)
    out['has_rainforest_label'] = labels_tags.str.contains(
        'rainforest|fairtrade|commerce-equitable', regex=True
    ).astype(int)
    out['has_hve_label'] = labels_tags.str.contains(
        'hve|high-environmental|valeur-environnementale', regex=True
    ).astype(int)
    out['n_eco_labels'] = (
        out['has_organic_label'] + out['has_rainforest_label'] + out['has_hve_label']
    )

    # ── 2. Origine → bonus/malus EPI (-7 à +3 pts) ────────────────────────
    origins_tags = df['origins_tags'].apply(safe_str)
    countries_tags = df['countries_tags'].apply(safe_str)
    combined_origin = origins_tags + ' ' + countries_tags

    # Pays à fort score EPI (bonus)
    top_epi_countries = ['france', 'germany', 'denmark', 'sweden', 'netherlands',
                         'austria', 'finland', 'switzerland', 'norway']
    out['origin_good_epi'] = combined_origin.apply(
        lambda x: int(any(c in x for c in top_epi_countries))
    )
    # Pays à faible score EPI (malus)
    bad_epi_countries = ['china', 'india', 'brazil', 'indonesia', 'vietnam', 'malaysia']
    out['origin_bad_epi'] = combined_origin.apply(
        lambda x: int(any(c in x for c in bad_epi_countries))
    )
    out['is_local_france'] = combined_origin.str.contains('france').astype(int)

    # ── 3. Poissons menacés → malus -10 pts ──────────────────────────────
    ingredients_text = df['ingredients_text'].apply(safe_str)
    out['has_endangered_fish'] = ingredients_text.apply(
        lambda x: int(any(f in x for f in ENDANGERED_FISH_KEYWORDS))
    )

    # ── 4. Emballage → malus circularité (0 à -10 pts) ───────────────────
    packaging_tags = df['packaging_tags'].apply(safe_str)
    out['packaging_recyclable'] = packaging_tags.apply(
        lambda x: int(any(k in x for k in ['recycl', 'compostable', 'reusable', 'glass', 'paper', 'cardboard']))
    )
    out['packaging_plastic'] = packaging_tags.apply(
        lambda x: int('plastic' in x or 'polystyrene' in x or 'pvc' in x)
    )
    out['packaging_score_proxy'] = out['packaging_recyclable'] - out['packaging_plastic']

    # ── 5. Catégorie produit (ACV Agribalyse proxy) ───────────────────────
    # La catégorie est le meilleur proxy du single score ACV
    pnns1 = df.get('pnns_groups_1', pd.Series([''] * len(df), index=df.index)).apply(safe_str)
    pnns2 = df.get('pnns_groups_2', pd.Series([''] * len(df), index=df.index)).apply(safe_str)

    # Groupes à fort impact ACV (viande rouge, produits transformés)
    out['is_meat_beef'] = (pnns1 + ' ' + pnns2 + ' ' + out['text_categories']).str.contains(
        r'beef|boeuf|veal|veau|lamb|agneau|pork|porc|processed\s*meat', regex=True
    ).astype(int)
    out['is_dairy'] = pnns1.str.contains('milk|dairy|fromage|cheese|lait', regex=True).astype(int)
    out['is_fish'] = (pnns1 + ' ' + pnns2).str.contains('fish|seafood|poisson|fruits\s*de\s*mer', regex=True).astype(int)
    out['is_plant_based'] = (pnns1 + ' ' + pnns2).str.contains(
        'vegetables|fruits|legumes|cereals|veg', regex=True
    ).astype(int)
    out['is_beverage'] = pnns1.str.contains('beverage|drink|boisson|jus|juice|milk', regex=True).astype(int)

    # ── 6. Score Agribalyse direct (si disponible) ────────────────────────
    if 'agribalyse_score' in df.columns:
        out['agribalyse_score'] = pd.to_numeric(df['agribalyse_score'], errors='coerce').fillna(-1)
    else:
        out['agribalyse_score'] = -1.0

    return out


print("⚙️  Feature engineering en cours...")
df_feat = extract_features(df_raw)
print(f"✅ Features extraites : {df_feat.shape}")
print("\nAperçu des features binaires :")
binary_cols = [c for c in df_feat.columns if c.startswith(('has_', 'is_', 'origin_', 'packaging_'))]
df_feat[binary_cols].mean().sort_values(ascending=False).round(3)

In [ ]:
# ── Nettoyage final ────────────────────────────────────────────────────────
df_feat = df_feat.dropna(subset=['ecoscore_score'])
df_feat['ecoscore_score'] = df_feat['ecoscore_score'].clip(0, 100)

# Mapper les grades en entiers (pour classification)
GRADE_MAP = {'a-plus': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6}
df_feat['grade_label'] = df_feat['ecoscore_grade'].map(GRADE_MAP)
df_feat = df_feat.dropna(subset=['grade_label'])
df_feat['grade_label'] = df_feat['grade_label'].astype(int)

print(f"✅ Dataset final : {len(df_feat):,} produits")
print("\nDistribution des grades :")
print(df_feat['ecoscore_grade'].value_counts().sort_index())

## 5. Génération des Embeddings

### 5.1 TF-IDF Vectorization (approche rapide)
Transforme le texte des produits en vecteurs de pondération TF-IDF

In [ ]:
# ── Split train/test ────────────────────────────────────────────────────────
y_reg   = df_feat['ecoscore_score'].values        # Régression : score numérique
y_clf   = df_feat['grade_label'].values           # Classification : grade 0-6

idx_train, idx_test = train_test_split(
    np.arange(len(df_feat)), test_size=0.2, random_state=RANDOM_STATE,
    stratify=y_clf
)

# ── TF-IDF sur le texte enrichi ────────────────────────────────────────────
print("🔤 Création des vecteurs TF-IDF...")

# Vectoriseur principal : nom + catégories + labels + ingrédients
tfidf_rich = TfidfVectorizer(
    max_features=30_000,
    min_df=5,
    max_df=0.95,
    ngram_range=(1, 2),   # unigrammes et bigrammes
    sublinear_tf=True,    # log(tf) pour réduire l'effet des mots très fréquents
    strip_accents='unicode',
    analyzer='word'
)

X_tfidf_rich_train = tfidf_rich.fit_transform(df_feat['text_rich'].iloc[idx_train])
X_tfidf_rich_test  = tfidf_rich.transform(df_feat['text_rich'].iloc[idx_test])

# Vectoriseur secondaire : emballage + pays (n-grams courts)
tfidf_meta = TfidfVectorizer(
    max_features=5_000,
    min_df=3,
    ngram_range=(1, 1),
    sublinear_tf=True,
    strip_accents='unicode'
)
text_meta = (df_feat['text_countries'] + ' ' + df_feat['text_packaging'] + ' ' + df_feat['text_origins'])
X_tfidf_meta_train = tfidf_meta.fit_transform(text_meta.iloc[idx_train])
X_tfidf_meta_test  = tfidf_meta.transform(text_meta.iloc[idx_test])

print(f"  Rich TF-IDF shape : {X_tfidf_rich_train.shape}")
print(f"  Meta TF-IDF shape : {X_tfidf_meta_train.shape}")

In [ ]:
# ── Features structurées (bonus/malus explicites) ──────────────────────────
STRUCT_COLS = [
    'has_organic_label', 'has_rainforest_label', 'has_hve_label', 'n_eco_labels',
    'origin_good_epi', 'origin_bad_epi', 'is_local_france',
    'has_endangered_fish',
    'packaging_recyclable', 'packaging_plastic', 'packaging_score_proxy',
    'is_meat_beef', 'is_dairy', 'is_fish', 'is_plant_based', 'is_beverage',
    'agribalyse_score',
]

X_struct = df_feat[STRUCT_COLS].fillna(0).values.astype(np.float32)
X_struct_train = csr_matrix(X_struct[idx_train])
X_struct_test  = csr_matrix(X_struct[idx_test])

# ── Assemblage final ────────────────────────────────────────────────────────
# TF-IDF rich + TF-IDF meta + features structurées
X_train_full = hstack([X_tfidf_rich_train, X_tfidf_meta_train, X_struct_train])
X_test_full  = hstack([X_tfidf_rich_test,  X_tfidf_meta_test,  X_struct_test])

y_reg_train = y_reg[idx_train]; y_reg_test = y_reg[idx_test]
y_clf_train = y_clf[idx_train]; y_clf_test = y_clf[idx_test]

print(f"✅ Matrice finale : train={X_train_full.shape}, test={X_test_full.shape}")

### 5.2 Sentence Transformers (embeddings sémantiques)

Utilise un modèle multilingue pré-entraîné pour capturer la sémantique des produits alimentaires.

In [ ]:
from sentence_transformers import SentenceTransformer

# Modèle léger multilingue adapté aux textes courts
# 'paraphrase-multilingual-MiniLM-L12-v2' : bon compromis vitesse/qualité
# Pour plus de précision : 'paraphrase-multilingual-mpnet-base-v2'
SBERT_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'

print(f"🤗 Chargement du modèle Sentence Transformers : {SBERT_MODEL}")
sbert = SentenceTransformer(SBERT_MODEL)

# On encode uniquement un sous-ensemble pour économiser du temps
# (augmenter SBERT_SAMPLE pour plus de précision)
SBERT_SAMPLE = min(50_000, len(df_feat))  # Augmenter pour plus de précision

# Texte court et informatif pour l'encodage
texts_to_encode = (
    df_feat['text_product'] + ' ' + df_feat['text_categories']
).iloc[:SBERT_SAMPLE].tolist()

# Textes vides → placeholder
texts_to_encode = [t if t.strip() else 'unknown food product' for t in texts_to_encode]

print(f"⚡ Encodage de {SBERT_SAMPLE:,} produits (batch_size=256)...")
embeddings = sbert.encode(
    texts_to_encode,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # normalisation L2 pour la similarité cosinus
)

print(f"✅ Embeddings shape : {embeddings.shape}  (dim={embeddings.shape[1]})")

In [ ]:
# ── Visualisation des embeddings (PCA 2D) ──────────────────────────────────
from sklearn.decomposition import PCA

GRADES_STR = ['A+', 'A', 'B', 'C', 'D', 'E', 'F']
GRADE_COLORS_VIZ = ['#006400', '#228B22', '#9ACD32', '#FFD700', '#FF8C00', '#DC143C', '#8B0000']

# PCA sur les 5000 premiers points
n_viz = min(5000, SBERT_SAMPLE)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
emb_2d = pca.fit_transform(embeddings[:n_viz])

grades_viz = y_clf[:n_viz]

fig, ax = plt.subplots(figsize=(12, 8))
for g_idx, (g_name, g_color) in enumerate(zip(GRADES_STR, GRADE_COLORS_VIZ)):
    mask = grades_viz == g_idx
    if mask.sum() > 0:
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                   c=g_color, label=f'Grade {g_name} (n={mask.sum():,})',
                   alpha=0.4, s=8, rasterized=True)

ax.set_title('Embeddings SBERT (PCA 2D) colorés par Green-Score Grade', fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(markerscale=3, loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('embeddings_pca.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Entraînement des Modèles

### 6.1 Modèle de Régression (score numérique 0-100)

In [ ]:
# ── 6.1 Régression : LightGBM sur TF-IDF + features structurées ──────────
print("🌲 Entraînement LightGBM Régression (TF-IDF + features) ...")

lgb_reg = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=63,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    min_child_samples=20,
    objective='regression',
    metric='mae',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

lgb_reg.fit(
    X_train_full, y_reg_train,
    eval_set=[(X_test_full, y_reg_test)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

pred_reg = lgb_reg.predict(X_test_full)
pred_reg = np.clip(pred_reg, 0, 100)

mae  = mean_absolute_error(y_reg_test, pred_reg)
rmse = np.sqrt(mean_squared_error(y_reg_test, pred_reg))
r2   = r2_score(y_reg_test, pred_reg)

print(f"\n📊 Régression — Résultats sur test set :")
print(f"  MAE  = {mae:.2f} points")
print(f"  RMSE = {rmse:.2f} points")
print(f"  R²   = {r2:.4f}")

### 6.2 Modèle de Classification (grade A+ → F)

In [ ]:
# ── 6.2 Classification : LightGBM multi-classe ───────────────────────────
print("🌲 Entraînement LightGBM Classification (grade A+ → F) ...")

lgb_clf = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=63,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    min_child_samples=20,
    objective='multiclass',
    num_class=7,
    metric='multi_logloss',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

lgb_clf.fit(
    X_train_full, y_clf_train,
    eval_set=[(X_test_full, y_clf_test)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

pred_clf = lgb_clf.predict(X_test_full)
acc = accuracy_score(y_clf_test, pred_clf)

print(f"\n📊 Classification — Résultats :")
print(f"  Accuracy = {acc:.4f} ({acc*100:.2f}%)")
print("\nRapport détaillé :")
print(classification_report(
    y_clf_test, pred_clf,
    target_names=GRADES_STR,
    zero_division=0
))

In [ ]:
# ── Matrice de confusion ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion matrix
cm = confusion_matrix(y_clf_test, pred_clf)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=GRADES_STR, yticklabels=GRADES_STR, ax=axes[0])
axes[0].set_title('Matrice de confusion (normalisée)', fontweight='bold')
axes[0].set_ylabel('Grade réel')
axes[0].set_xlabel('Grade prédit')

# Scatter : score prédit vs réel
axes[1].scatter(y_reg_test[:5000], pred_reg[:5000],
                alpha=0.2, s=5, color='steelblue', rasterized=True)
axes[1].plot([0, 100], [0, 100], 'r--', lw=2, label='Prédiction parfaite')
axes[1].set_title(f'Score réel vs prédit (MAE={mae:.2f})', fontweight='bold')
axes[1].set_xlabel('Score réel')
axes[1].set_ylabel('Score prédit')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_results.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 Modèle basé sur Sentence Embeddings (SBERT)

In [ ]:
# ── 6.3 Régression SBERT + LightGBM ────────────────────────────────────
# On utilise le sous-ensemble encodé par SBERT
print("🤖 Entraînement LightGBM sur Sentence Embeddings (SBERT)...")

sbert_idx = np.arange(SBERT_SAMPLE)
sbert_y   = y_reg[:SBERT_SAMPLE]
sbert_clf_y = y_clf[:SBERT_SAMPLE]

# Concaténer les embeddings SBERT avec les features structurées
X_struct_sbert = X_struct[:SBERT_SAMPLE]
X_sbert_full   = np.hstack([embeddings, X_struct_sbert])

s_train, s_test = train_test_split(
    np.arange(SBERT_SAMPLE), test_size=0.2,
    random_state=RANDOM_STATE, stratify=sbert_clf_y
)

lgb_sbert = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    objective='regression',
    random_state=RANDOM_STATE,
    n_jobs=-1, verbose=-1
)
lgb_sbert.fit(X_sbert_full[s_train], sbert_y[s_train],
              eval_set=[(X_sbert_full[s_test], sbert_y[s_test])],
              callbacks=[lgb.early_stopping(30, verbose=False)])

pred_sbert = lgb_sbert.predict(X_sbert_full[s_test])
pred_sbert = np.clip(pred_sbert, 0, 100)

mae_s  = mean_absolute_error(sbert_y[s_test], pred_sbert)
r2_s   = r2_score(sbert_y[s_test], pred_sbert)
print(f"\n📊 SBERT + LightGBM :")
print(f"  MAE = {mae_s:.2f} | R² = {r2_s:.4f}")
print(f"\n📊 Comparaison :")
print(f"  TF-IDF + LightGBM : MAE={mae:.2f} | R²={r2:.4f}")
print(f"  SBERT  + LightGBM : MAE={mae_s:.2f} | R²={r2_s:.4f}")

## 7. Analyse des Features Importantes

Quelles informations textuelles et structurées influencent le plus le Green-Score ?

In [ ]:
# ── Top features TF-IDF ───────────────────────────────────────────────────
n_tfidf_rich = X_tfidf_rich_train.shape[1]
n_tfidf_meta = X_tfidf_meta_train.shape[1]
n_struct     = len(STRUCT_COLS)

feature_names = (
    [f'rich_{v}' for v in tfidf_rich.get_feature_names_out()] +
    [f'meta_{v}' for v in tfidf_meta.get_feature_names_out()] +
    STRUCT_COLS
)

fi = lgb_reg.feature_importances_
fi_df = pd.DataFrame({'feature': feature_names[:len(fi)], 'importance': fi})
fi_df = fi_df.sort_values('importance', ascending=False)

# Séparer les features textuelles des features structurées
fi_struct = fi_df[fi_df['feature'].isin(STRUCT_COLS)]
fi_text   = fi_df[~fi_df['feature'].isin(STRUCT_COLS)].head(25)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Features structurées
axes[0].barh(fi_struct['feature'], fi_struct['importance'], color='#2ecc71')
axes[0].set_title('Importance — Features structurées (Green-Score)', fontweight='bold')
axes[0].set_xlabel('Importance LightGBM')
axes[0].invert_yaxis()

# Top tokens TF-IDF
fi_text_clean = fi_text.copy()
fi_text_clean['feature'] = fi_text_clean['feature'].str.replace('^rich_|^meta_', '', regex=True)
axes[1].barh(fi_text_clean['feature'], fi_text_clean['importance'], color='steelblue')
axes[1].set_title('Top 25 tokens TF-IDF les plus importants', fontweight='bold')
axes[1].set_xlabel('Importance LightGBM')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Recherche par Similarité Vectorielle (FAISS)

Trouve les produits les plus similaires à un produit donné et prédit leur Green-Score.

In [ ]:
try:
    import faiss
    HAS_FAISS = True
except ImportError:
    print("⚠️  FAISS non installé. Installation...")
    os.system('pip install faiss-cpu -q')
    import faiss
    HAS_FAISS = True

print("🔍 Construction de l'index FAISS (cosine similarity)...")

# Embeddings déjà normalisés L2 → inner product = cosine similarity
d = embeddings.shape[1]  # dimension des vecteurs
index = faiss.IndexFlatIP(d)  # Inner Product (= cosine si vecteurs normalisés)
index.add(embeddings.astype(np.float32))

print(f"✅ Index FAISS construit avec {index.ntotal:,} vecteurs (dim={d})")


def search_similar_products(query_text: str, k: int = 5) -> pd.DataFrame:
    """Cherche les k produits les plus similaires et affiche leur Green-Score prédit."""
    # Encoder la requête
    q_emb = sbert.encode([query_text], normalize_embeddings=True).astype(np.float32)

    # Recherche dans l'index FAISS
    scores, indices = index.search(q_emb, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = df_feat.iloc[idx]
        results.append({
            'product_name'      : df_raw.iloc[idx].get('product_name', 'N/A'),
            'similarity'        : round(float(score), 4),
            'ecoscore_grade'    : row['ecoscore_grade'],
            'ecoscore_score'    : round(row['ecoscore_score'], 1),
            'categories'        : str(df_raw.iloc[idx].get('categories', ''))[:60],
            'has_organic_label' : bool(row['has_organic_label']),
        })
    return pd.DataFrame(results)


# Test
print("\n🔍 Produits similaires à 'yaourt bio aux fraises France' :")
search_similar_products('yaourt bio aux fraises France', k=5)

## 9. Inférence — Prédire le Green-Score d'un nouveau produit

In [ ]:
GRADE_THRESHOLDS = [
    (80, 'A+'), (70, 'A'), (55, 'B'), (40, 'C'), (25, 'D'), (10, 'E'), (0, 'F')
]
GRADE_EMOJIS = {
    'A+': '🟢🟢', 'A': '🟢', 'B': '🟡🟢', 'C': '🟡',
    'D': '🟠', 'E': '🔴', 'F': '🔴🔴'
}

def score_to_grade(score):
    for threshold, grade in GRADE_THRESHOLDS:
        if score >= threshold:
            return grade
    return 'F'


def predict_ecoscore(product: dict) -> dict:
    """
    Prédit le Green-Score d'un produit.

    Paramètres attendus (tous optionnels) :
      - product_name   : str
      - categories     : str
      - ingredients    : str
      - labels         : str  (ex: 'bio, rainforest alliance')
      - countries      : str  (ex: 'France')
      - packaging      : str  (ex: 'verre, recyclable')
    """
    # Construire le texte enrichi
    text_rich = ' '.join([
        product.get('product_name', ''),
        product.get('categories', ''),
        product.get('labels', ''),
        product.get('ingredients', '')[:300]
    ]).lower().strip()

    text_meta = ' '.join([
        product.get('countries', ''),
        product.get('packaging', ''),
    ]).lower()

    # Vecteurs TF-IDF
    v_rich = tfidf_rich.transform([text_rich])
    v_meta = tfidf_meta.transform([text_meta])

    # Features structurées
    labels_lower = product.get('labels', '').lower()
    ingr_lower   = product.get('ingredients', '').lower()
    pack_lower   = product.get('packaging', '').lower()
    ctry_lower   = product.get('countries', '').lower()

    struct = {
        'has_organic_label'   : int(any(k in labels_lower for k in ['bio', 'organic', 'biologique'])),
        'has_rainforest_label': int('rainforest' in labels_lower),
        'has_hve_label'       : int('hve' in labels_lower),
        'n_eco_labels'        : sum([int(any(k in labels_lower for k in ['bio','organic'])),
                                     int('rainforest' in labels_lower), int('hve' in labels_lower)]),
        'origin_good_epi'     : int(any(c in ctry_lower for c in ['france','allemagne','germany','suède','sweden'])),
        'origin_bad_epi'      : int(any(c in ctry_lower for c in ['china','inde','india','bresil','brazil'])),
        'is_local_france'     : int('france' in ctry_lower),
        'has_endangered_fish' : int(any(f in ingr_lower for f in ENDANGERED_FISH_KEYWORDS)),
        'packaging_recyclable': int(any(k in pack_lower for k in ['recycl','verre','glass','carton','papier'])),
        'packaging_plastic'   : int(any(k in pack_lower for k in ['plastique','plastic','polystyrene'])),
        'packaging_score_proxy': 0,
        'is_meat_beef'        : int(any(k in ingr_lower for k in ['boeuf','beef','veau','agneau'])),
        'is_dairy'            : int(any(k in ingr_lower for k in ['lait','milk','fromage','cheese'])),
        'is_fish'             : int(any(k in ingr_lower for k in ['poisson','fish','saumon','salmon'])),
        'is_plant_based'      : int(any(k in ingr_lower for k in ['légumes','vegetables','fruits','céréales'])),
        'is_beverage'         : int(any(k in product.get('categories','').lower() for k in ['boisson','beverage','drink'])),
        'agribalyse_score'    : -1.0,
    }
    struct['packaging_score_proxy'] = struct['packaging_recyclable'] - struct['packaging_plastic']

    v_struct = csr_matrix([[struct[c] for c in STRUCT_COLS]])
    X_new    = hstack([v_rich, v_meta, v_struct])

    # Prédictions
    score_pred = float(np.clip(lgb_reg.predict(X_new)[0], 0, 100))
    grade_idx  = int(lgb_clf.predict(X_new)[0])
    grade_pred = GRADES_STR[grade_idx]
    grade_from_score = score_to_grade(score_pred)

    # Explication du score (composantes)
    explanation = []
    if struct['has_organic_label']:    explanation.append('🌿 Label bio (+10 à +20 pts)')
    if struct['has_rainforest_label']: explanation.append('🌳 Rainforest Alliance (+10 pts)')
    if struct['is_local_france']:      explanation.append('🇫🇷 Origine France (bonus EPI)')
    if struct['origin_bad_epi']:       explanation.append('⚠️  Origine à faible score EPI (malus)')
    if struct['has_endangered_fish']:  explanation.append('🐟 Espèce menacée (-10 pts)')
    if struct['packaging_plastic']:    explanation.append('♻️  Emballage plastique (malus)')
    if struct['packaging_recyclable']: explanation.append('✅ Emballage recyclable (bonus)')
    if struct['is_meat_beef']:         explanation.append('🥩 Viande rouge (fort impact ACV)')

    return {
        'score'           : round(score_pred, 1),
        'grade_predicted' : grade_pred,
        'grade_from_score': grade_from_score,
        'emoji'           : GRADE_EMOJIS.get(grade_from_score, '❓'),
        'explanation'     : explanation
    }


# ── Tests d'inférence ────────────────────────────────────────────────────
test_products = [
    {
        'product_name': 'Yaourt bio aux fraises',
        'categories'  : 'Produits laitiers, Yaourts',
        'ingredients' : 'lait entier, fraises, sucre de canne',
        'labels'      : 'Agriculture Biologique, AB',
        'countries'   : 'France',
        'packaging'   : 'pot en verre recyclable'
    },
    {
        'product_name': 'Steak haché congelé',
        'categories'  : 'Viandes, Boeuf, Surgelés',
        'ingredients' : 'boeuf haché 100%',
        'labels'      : '',
        'countries'   : 'Brazil',
        'packaging'   : 'barquette plastique non recyclable'
    },
    {
        'product_name': 'Thon en boîte à l\'huile',
        'categories'  : 'Conserves, Poissons',
        'ingredients' : 'thon, huile d\'olive',
        'labels'      : '',
        'countries'   : 'Thailand',
        'packaging'   : 'boîte métal recyclable'
    },
    {
        'product_name': 'Lentilles vertes du Puy AOP',
        'categories'  : 'Légumineuses, Légumes secs',
        'ingredients' : 'lentilles vertes 100%',
        'labels'      : 'AOP, Agriculture raisonnée, HVE',
        'countries'   : 'France',
        'packaging'   : 'sachet papier recyclable'
    },
]

print("🌿 Prédictions Green-Score\n" + "="*60)
for p in test_products:
    result = predict_ecoscore(p)
    print(f"\n📦 {p['product_name']}")
    print(f"   Score : {result['score']}/100  |  Grade : {result['emoji']} {result['grade_from_score']}")
    if result['explanation']:
        print("   Facteurs détectés :")
        for e in result['explanation']:
            print(f"     {e}")

## 10. Analyse par Catégorie de Produit

In [ ]:
# Score moyen par groupe PNNS
if 'pnns_groups_1' in df_raw.columns:
    df_analysis = df_raw[['pnns_groups_1']].join(df_feat[['ecoscore_score']])
    df_analysis = df_analysis.dropna(subset=['pnns_groups_1', 'ecoscore_score'])
    df_analysis = df_analysis[df_analysis['pnns_groups_1'] != 'unknown']

    cat_scores = (
        df_analysis.groupby('pnns_groups_1')['ecoscore_score']
        .agg(['mean', 'count'])
        .query('count >= 100')
        .sort_values('mean', ascending=True)
    )

    fig, ax = plt.subplots(figsize=(12, max(6, len(cat_scores)*0.4)))
    colors = ['#a00000' if s < 40 else ('#FF8C00' if s < 55 else
              ('#FFD700' if s < 70 else '#228B22'))
              for s in cat_scores['mean']]
    bars = ax.barh(cat_scores.index, cat_scores['mean'], color=colors)
    ax.set_title('Score Green-Score moyen par catégorie alimentaire', fontsize=14, fontweight='bold')
    ax.set_xlabel('Green-Score moyen /100')
    ax.axvline(55, color='gray', linestyle='--', alpha=0.5, label='Grade B')
    ax.axvline(40, color='orange', linestyle='--', alpha=0.5, label='Grade C')
    ax.legend(loc='lower right')
    for bar, (_, row) in zip(bars, cat_scores.iterrows()):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f"{row['mean']:.1f} (n={int(row['count']):,})",
                va='center', fontsize=8)
    plt.tight_layout()
    plt.savefig('score_by_category.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠️  Colonne pnns_groups_1 non disponible")

## 11. Sauvegarde des Modèles

In [ ]:
import pickle

MODELS_DIR = Path('models')
MODELS_DIR.mkdir(exist_ok=True)

# Sauvegarder le pipeline complet
pipeline = {
    'tfidf_rich'  : tfidf_rich,
    'tfidf_meta'  : tfidf_meta,
    'lgb_reg'     : lgb_reg,    # Régression score
    'lgb_clf'     : lgb_clf,    # Classification grade
    'lgb_sbert'   : lgb_sbert,  # Modèle sur embeddings SBERT
    'struct_cols' : STRUCT_COLS,
    'grades'      : GRADES_STR,
    'grade_map'   : GRADE_MAP,
    'metrics'     : {
        'regression_mae'  : mae,
        'regression_r2'   : r2,
        'classification_accuracy': acc,
        'sbert_mae'       : mae_s,
        'sbert_r2'        : r2_s,
    }
}

with open(MODELS_DIR / 'ecoscore_pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

# Sauvegarder aussi les embeddings SBERT pour réutilisation
np.save(MODELS_DIR / 'sbert_embeddings.npy', embeddings)
sbert.save(str(MODELS_DIR / 'sbert_model'))

print("✅ Modèles sauvegardés dans models/")
print("\n📊 Résumé des performances :")
print(f"  TF-IDF + LightGBM Régression : MAE={mae:.2f} pts, R²={r2:.4f}")
print(f"  TF-IDF + LightGBM Classification : Accuracy={acc*100:.2f}%")
print(f"  SBERT + LightGBM Régression : MAE={mae_s:.2f} pts, R²={r2_s:.4f}")

## Conclusion

### Résultats obtenus

| Modèle | Tâche | Métrique |
|--------|-------|----------|
| **TF-IDF + LightGBM** | Régression (score /100) | MAE ≈ X pts, R² ≈ X |
| **TF-IDF + LightGBM** | Classification (A+ → F) | Accuracy ≈ X% |
| **SBERT + LightGBM** | Régression (score /100) | MAE ≈ X pts, R² ≈ X |

### Facteurs les plus prédictifs
1. **Catégorie du produit** (proxy ACV Agribalyse) — la feature la plus importante
2. **Labels environnementaux** (bio, Rainforest Alliance, HVE)
3. **Origine géographique** (score EPI du pays)
4. **Type d'emballage** (plastique vs recyclable)
5. **Présence d'espèces menacées** (thon, espadon...)

### Pistes d'amélioration
- 🔬 Intégrer la base Agribalyse directement pour les single scores ACV
- 🌡️ Ajouter la saisonnalité (date de consommation × ingrédients frais)
- 🤖 Fine-tuner un modèle de langage (CamemBERT/FoodBERT) sur les données OpenFoodFacts
- 📦 Enrichir les données d'emballage avec l'index de circularité complet
- 🗺️ Mapper précisément les pays vers leur score EPI (Environmental Performance Index)